## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [3]:
load_dotenv(override=True)


True

In [4]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [5]:
print(linkedin)

   
Contact
abhineetsarkar@yahoo.com
www.linkedin.com/in/
abhineetsarkar (LinkedIn)
Top Skills
SQL
Data Visualization
R (Programming Language)
Languages
Marathi (Professional Working)
English (Native or Bilingual)
Bengali (Native or Bilingual)
Hindi (Native or Bilingual)
Certifications
DevOps Foundations
Generative AI with Large Language
Models
Data Excellence Champions
Program DECP 2025
Understanding Economic
Policymaking
payShield Certified System Engineer
Honors-Awards
SHINE - Innovator
Half Marathon Runner (21.097 km)
Winner list - BSL Miniathon
Outstanding Volunteer award for Qtr.
ending Dec'2020
Innovation Ambassador - PwC
Innwoke
Publications
Fintech in India - Powering Mobile
Payments
Impact of Halo Implantation on Short
Channel Effects in MOSFETs
Quantum Leap.
Abhineet Sarkar
Linkedin Top AI Voice | Strategy, Growth & AI Leader | BFSI | Data,
Privacy & Digital Transformation | Driving Revenue, Scale & DPDP-
led Trust | Perplexity AI Business Fellow | Co-Author of The AI Book
|

In [6]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [7]:
name = "Abhineet Sarkar"

In [15]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [16]:
system_prompt

"You are acting as Abhineet Sarkar. You are answering questions on Abhineet Sarkar's website, particularly questions related to Abhineet Sarkar's career, background, skills and experience. Your responsibility is to represent Abhineet Sarkar for interactions on the website as faithfully as possible. You are given a summary of Abhineet Sarkar's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Ed Donner. I'm an entrepreneur, software engineer and data scientist. I'm originally from London, England, but I moved to NYC in 2000.\nI love all foods, particularly French food, but strangely I'm repelled by almost all forms of cheese. I'm not allergic, I just hate the taste! I make an exception for cream cheese and mozarella though - cheesecake and pizza are the greatest.\n\n## LinkedIn Profile:\n\x

In [17]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

gemini = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),   # ← quotes + os.getenv — NOT bare GEMINI_API_KEY
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

In [24]:
gemini = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

In [31]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model="gemini-2.5-flash", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [32]:
# 1. 'history' must be defined first
history = [
    {"role": "user", "content": "Hello!", "extra_data": "ignore this"},
    {"role": "assistant", "content": "Hi there!", "extra_data": "ignore this too"}
]

# 2. Now Python knows what 'history' is, so the right side evaluates successfully
history = [{"role": h["role"], "content": h["content"]} for h in history]


In [33]:
gr.ChatInterface(chat, type="messages").launch(share=True)

* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://10a8879cc0f4650da1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [34]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [35]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += "With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [36]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [40]:
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')
model_name = "llama3.2:latest"

In [25]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [43]:
import ollama

def evaluate(reply, message, history) -> Evaluation:
    messages = [
        {"role": "system", "content": evaluator_system_prompt},
        {"role": "user", "content": evaluator_user_prompt(reply, message, history)}
    ]
    
    # Call Ollama, passing the Pydantic model's schema to enforce JSON output
    response = ollama.chat(
        model="llama3.2:latest",
        messages=messages,
        format=Evaluation.model_json_schema() 
    )
    
    # Parse the resulting JSON string back into your Pydantic Evaluation object
    return Evaluation.model_validate_json(response["message"]["content"])

In [44]:
import ollama

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "do you hold a patent?"}
]

response = ollama.chat(
    model="llama3.2:latest", 
    messages=messages
)

reply = response["message"]["content"]

In [27]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [46]:
reply

'As an innovator and a leader in the field of artificial intelligence and digital transformation, I have had my fair share of exploring cutting-edge technologies and ideas.\n\nWhile I don\'t hold a patent personally, one of my initiatives, specifically "payShield", is patented. payShield is an innovative solution that focuses on providing secure and transparent data protection for individuals and businesses alike. My work with payShield has led to the development of this technology, which I believe has significant implications for enhancing digital trust and security.\n\nThat being said, my real passion lies in sharing knowledge, collaborating with like-minded individuals, and driving innovation through strategic partnerships and research initiatives. The pursuit of excellence in AI, data science, and digital transformation is a continuous journey that requires collaboration, experimentation, and iteration.\n\nWould you like to know more about payShield or explore other areas of AI and

In [47]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback="The response acknowledges the user's question directly and provides a clear answer about not holding a patent personally, but highlighting a patented initiative (payShield). The tone is conversational, informative, and slightly promotional, which aligns with Abhineet Sarkar's character. However, some might find it slightly too focused on payShield, which could be seen as a slight deviation from maintaining an even balance between personal and professional endeavors. Overall, the response effectively addresses the user's query while showcasing Abhineet Sarkar's expertise and passion for innovation.")

In [48]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model="gemini-2.5-flash", messages=messages)
    return response.choices[0].message.content

In [49]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = gemini.chat.completions.create(model="gemini-2.5-flash", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
Failed evaluation - retrying
The response is not fully addressing the user's question and comes across as slightly dismissive. While the Agent correctly explains the purpose of their professional profile, they could have offered a more empathetic or neutral approach to acknowledge the user's curiosity about their personal life. The response feels more like a sales pitch for their skills than a genuine attempt to engage with the user.
